# Supplementary Experiment: Per-Protein Min-Max PAE Normalization
A self-contained, standalone robustness check on the project's central negative result (PAE-weighting via `1/(1+PAE)` does not beat an unweighted GATv2 ablation). This notebook tests whether a **different** edge-weight transform, one specifically designed to fix the saturation problem diagnosed in Phase 4 Section 2, changes that conclusion.

**Does not modify any existing notebook, checkpoint, or result file.** Every output this notebook produces is written under `pae_minmax_experiment/`. All existing project files (`protein_graphs/`, `models/`, `phase4_*.csv`, `phase5_*.csv`, `clinvar_graph_ready.csv`, `best_config.json`) are only ever **read**, never written to.

### The problem this addresses
The original edge weight, `1/(1+PAE)`, is a *global* transform: every protein's PAE values get mapped through the same fixed curve. Phase 4 Section 2 showed this saturates hard, the 50th, 75th, 95th and 99th percentiles of edge weight across ~1.68M sampled edges were *all* exactly 0.5, because (a) AlphaFold almost never reports PAE below ~1 Å for two *different* residues, capping the practical ceiling at `1/(1+1)=0.5`, and (b) contacts that pass the 8 Å cutoff are already the pairs AlphaFold tends to be confident about.

### The fix tested here: per-protein min-max normalization
Instead of a fixed global curve, each protein's own contact-edge PAE values are rescaled *relative to that protein's own range*: its most-confident edge always maps to weight 1.0, its least-confident edge always maps to weight 0.0, regardless of the protein's absolute PAE scale. This guarantees the full [0, 1] range is actually used for every single protein, directly targeting the saturation problem rather than just hoping it goes away.

```
pae_sym   = average(PAE[i][j], PAE[j][i])           # same symmetrization as the original transform
new_weight = 1 - (pae_sym - min(pae_sym in this protein)) / (max(pae_sym in this protein) - min(...))
```

**Inputs (all read-only):**
- `clinvar_graph_ready.csv`, `protein_graphs/{UniProt_ID}.pt`: topology, node features, and original distances are reused unchanged; only the PAE-weight edge-feature column is recomputed
- `alphafold_data/pae/{UniProt_ID}.json`: raw PAE matrices (already cached from Phase 2)
- `best_config.json`: the project's winning tuned hyperparameters (`B_reg_small`)
- `models/gatv2_unweighted_best.pt`, `models/gatv2_unweighted_final_best.pt`: existing unweighted ablation checkpoints, reloaded for cheap inference only (never retrained, see Section 8 for why this is valid)
- `phase5_gnn_predictions.csv`, `alphamissense_test_scores.csv`: cached GraphSAGE / AlphaMissense predictions, reused for significance testing without rerunning them

**Outputs (all new, under `pae_minmax_experiment/`):**
- `protein_graphs_minmax/{UniProt_ID}.pt`: 6,597 rebuilt graphs, new PAE-weight column only
- `models/gatv2_pae_minmax_default_best.pt`, `models/gatv2_pae_minmax_tuned_best.pt`: 2 new checkpoints
- `training_history_*.csv`, `figures/training_curves_*.png`: training curves for both new models
- `saturation_diagnostic.csv`, `figures/saturation_before_after.png`: proof the transform worked, checked *before* the expensive training runs
- `results_comparison.csv`: full model comparison table (new models + existing baselines)
- `significance_tests.csv`: paired bootstrap / DeLong / McNemar tests
- `pae_value_quantification_minmax.csv`: the proposal's %-change-in-AUPRC deliverable, for this transform
- `alphamissense_comparison_results.csv`, `alphamissense_significance_tests.csv`, `figures/alphamissense_comparison_minmax.png`: head-to-head vs. AlphaMissense
- `predictions_cache.csv`: every model's row-level test-set probabilities, cached for resumability


In [ ]:
import os, json, gzip, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import Sampler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from scipy import stats
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                              recall_score, f1_score, accuracy_score)
from statsmodels.stats.contingency_tables import mcnemar

from missense_dataset import MissenseVariantGraphDataset, NODE_FEATURE_DIM, ESM2_EMBEDDING_DIM, N_AA
from models import GATv2Classifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# --- Read-only sources from the existing project (never written to) ---
OLD_GRAPH_DIR   = "protein_graphs"
OLD_MODELS_DIR  = "models"
PAE_DIR         = os.path.join("alphafold_data", "pae")
BEST_CONFIG_PATH = "best_config.json"
PHASE5_PRED_CACHE = "phase5_gnn_predictions.csv"
AM_CACHE_PATH = "alphamissense_test_scores.csv"

# --- New, isolated output tree for this experiment only ---
EXPERIMENT_DIR = "pae_minmax_experiment"
NEW_GRAPH_DIR  = os.path.join(EXPERIMENT_DIR, "protein_graphs_minmax")
NEW_MODELS_DIR = os.path.join(EXPERIMENT_DIR, "models")
NEW_FIGURES_DIR = os.path.join(EXPERIMENT_DIR, "figures")
for d in (EXPERIMENT_DIR, NEW_GRAPH_DIR, NEW_MODELS_DIR, NEW_FIGURES_DIR):
    os.makedirs(d, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)


## 1. Load Phase 3 Output (read-only)
Same dataset used by every other phase, this notebook adds a new edge-weighting scheme, it does not touch the underlying variant table.


In [ ]:
df = pd.read_csv("clinvar_graph_ready.csv")
print(f"Total variants : {len(df):,}")
print(f"Unique proteins: {df['UniProt_ID'].nunique():,}")
print(f"  Pathogenic (1): {(df['Target']==1).sum():,}")
print(f"  Benign     (0): {(df['Target']==0).sum():,}")


## 2. The New Edge-Weight Transform
`minmax_pae_weight` takes the symmetrized PAE values for **one protein's contact edges** (the same edge population used everywhere else, the 8 Å Cα-Cα cutoff from Phase 3 is not changed, only how confidence gets turned into a number) and rescales them so the most confident edge in *that protein* maps to 1.0 and the least confident maps to 0.0.

**Edge case:** if a protein's edges all have (numerically) the same PAE, e.g. a very small structure with too few contacts to show variation, min-max normalization is undefined (division by zero). We fall back to weight = 1.0 for all of that protein's edges in that case: there is no evidence of *differential* confidence to encode, so treating them as uniformly confident is the same assumption a binary contact graph already makes.


In [ ]:
def minmax_pae_weight(pae_sym):
    """Per-protein min-max normalization of symmetrized PAE, inverted so 1.0 = most confident."""
    lo, hi = float(pae_sym.min()), float(pae_sym.max())
    if hi - lo < 1e-8:
        return np.ones_like(pae_sym, dtype=np.float32)
    return (1.0 - (pae_sym - lo) / (hi - lo)).astype(np.float32)


# --- Quick sanity check on one real protein before touching all 6,597 ---
example_id = "P38398"  # BRCA1 - the worked example used throughout Phases 2-3
example_graph = torch.load(os.path.join(OLD_GRAPH_DIR, f"{example_id}.pt"), weights_only=False)
with open(os.path.join(PAE_DIR, f"{example_id}.json")) as f:
    example_pae_matrix = np.array(json.load(f)[0]["predicted_aligned_error"], dtype=np.float32)

src, dst = example_graph.edge_index.numpy()
example_pae_sym = (example_pae_matrix[src, dst] + example_pae_matrix[dst, src]) / 2.0
old_weights_example = example_graph.edge_attr[:, 1].numpy()
new_weights_example = minmax_pae_weight(example_pae_sym)

print(f"{example_id} (BRCA1): {len(old_weights_example):,} edges\n")
print("OLD (1/(1+PAE))  weight -> min={:.4f} max={:.4f} mean={:.4f} std={:.4f}".format(
    old_weights_example.min(), old_weights_example.max(), old_weights_example.mean(), old_weights_example.std()))
print("NEW (min-max)    weight -> min={:.4f} max={:.4f} mean={:.4f} std={:.4f}".format(
    new_weights_example.min(), new_weights_example.max(), new_weights_example.mean(), new_weights_example.std()))
for p in [1, 25, 50, 75, 95, 99]:
    print(f"  p{p:2d}:  old={np.percentile(old_weights_example, p):.4f}   new={np.percentile(new_weights_example, p):.4f}")


## 3. Rebuild All Protein Graphs With the New Transform
Reuses each protein's already-cached graph for topology, node features (ESM-2 + wild-type one-hot), and Cα-Cα distances, **only the PAE-weight column of `edge_attr` is recomputed**. No PDB re-parsing, no ESM-2 recomputation, no re-downloading: this only needs the existing cached graph plus the existing cached PAE JSON, so it is fast (I/O + a few numpy ops per protein) rather than a full Phase-3 rerun.

Resumable exactly like Phases 2-3: a protein already present in `protein_graphs_minmax/` is skipped, so an interrupted run can simply be re-executed from the top.


In [ ]:
def rebuild_graph_minmax(uniprot_id):
    out_path = os.path.join(NEW_GRAPH_DIR, f"{uniprot_id}.pt")
    if os.path.exists(out_path):
        return "cached"

    old_path = os.path.join(OLD_GRAPH_DIR, f"{uniprot_id}.pt")
    pae_path = os.path.join(PAE_DIR, f"{uniprot_id}.json")
    if not (os.path.exists(old_path) and os.path.exists(pae_path)):
        return "missing_source_file"

    g = torch.load(old_path, weights_only=False)
    with open(pae_path) as f:
        pae_matrix = np.array(json.load(f)[0]["predicted_aligned_error"], dtype=np.float32)

    src, dst = g.edge_index.numpy()
    if pae_matrix.shape[0] != g.x.shape[0]:
        return f"pae_shape_mismatch (pae={pae_matrix.shape[0]}, graph={g.x.shape[0]})"

    pae_sym = (pae_matrix[src, dst] + pae_matrix[dst, src]) / 2.0
    new_weight = minmax_pae_weight(pae_sym)
    distances = g.edge_attr[:, 0].numpy()  # unchanged from the original graph
    new_edge_attr = torch.tensor(np.stack([distances, new_weight], axis=1), dtype=torch.float)

    new_g = Data(x=g.x, edge_index=g.edge_index, edge_attr=new_edge_attr,
                 uniprot_id=uniprot_id, num_residues=g.num_residues)
    torch.save(new_g, out_path)
    return "success"


unique_proteins = df["UniProt_ID"].unique()
rebuild_status = {}
for uid in tqdm(unique_proteins, desc="Rebuilding graphs (min-max PAE)"):
    rebuild_status[uid] = rebuild_graph_minmax(uid)

status_counts = pd.Series(rebuild_status).value_counts()
print("\nGraph rebuild status:")
print(status_counts.to_string())
failed = {k: v for k, v in rebuild_status.items() if v not in ("cached", "success")}
if failed:
    print(f"\nWARNING: {len(failed)} proteins failed to rebuild; their variants will be dropped "
          f"from this experiment's dataset below.")


## 4. Diagnostic: Did This Actually Fix the Saturation Problem?
Checked **before** spending ~14 hours training, on the same 300-protein random sample (seed=0) used by the original Phase 4 Section 2 diagnostic, so the OLD-transform numbers below are directly comparable to that earlier result (p50/p75/p95/p99 all exactly 0.5).


In [ ]:
rng_diag = np.random.RandomState(0)
sample_proteins_diag = rng_diag.choice(df["UniProt_ID"].unique(), size=300, replace=False)

old_weights_all, new_weights_all = [], []
for uid in tqdm(sample_proteins_diag, desc="Sampling old vs new PAE weights"):
    old_g = torch.load(os.path.join(OLD_GRAPH_DIR, f"{uid}.pt"), weights_only=False)
    new_g = torch.load(os.path.join(NEW_GRAPH_DIR, f"{uid}.pt"), weights_only=False)
    old_weights_all.append(old_g.edge_attr[:, 1].numpy())
    new_weights_all.append(new_g.edge_attr[:, 1].numpy())
old_weights_all = np.concatenate(old_weights_all)
new_weights_all = np.concatenate(new_weights_all)

print(f"Sampled {len(sample_proteins_diag)} proteins, {len(old_weights_all):,} total edges\n")
diag_rows = []
print(f"{'stat':>8s}  {'OLD (1/(1+PAE))':>18s}  {'NEW (min-max)':>15s}")
for p in [1, 5, 25, 50, 75, 95, 99]:
    o, n = np.percentile(old_weights_all, p), np.percentile(new_weights_all, p)
    print(f"p{p:6d}  {o:18.4f}  {n:15.4f}")
    diag_rows.append({"stat": f"p{p}", "old_transform": o, "new_transform": n})
diag_rows.append({"stat": "mean", "old_transform": old_weights_all.mean(), "new_transform": new_weights_all.mean()})
diag_rows.append({"stat": "std", "old_transform": old_weights_all.std(), "new_transform": new_weights_all.std()})
print(f"\n{'mean':>8s}  {old_weights_all.mean():18.4f}  {new_weights_all.mean():15.4f}")
print(f"{'std':>8s}  {old_weights_all.std():18.4f}  {new_weights_all.std():15.4f}")

pd.DataFrame(diag_rows).to_csv(os.path.join(EXPERIMENT_DIR, "saturation_diagnostic.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(old_weights_all, bins=50, alpha=0.55, label="OLD: 1/(1+PAE)", color="#B5493D")
ax.hist(new_weights_all, bins=50, alpha=0.55, label="NEW: per-protein min-max", color="#2E6F8E")
ax.set_xlabel("Edge weight"); ax.set_ylabel("Count (sampled edges)")
ax.set_title("Edge-weight distribution: old vs. new PAE transform")
ax.legend()
fig.savefig(os.path.join(NEW_FIGURES_DIR, "saturation_before_after.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("\nInterpretation: if the new-transform percentiles above are spread out (not identical "
      "to each other), the saturation problem has been fixed for this sample, the model "
      "training below can now actually test whether PAE-weighting helps once it has a usable signal.")


## 5. Train/Val/Test Split, Batching, Class Weighting
Identical logic and identical seed (42) to Phase 4/5, this is the same protein-level split, reproduced here (not re-imported) so this notebook has zero import-time dependency on the other notebooks having been run. Node counts, and therefore the split itself, are identical since it only depends on `clinvar_graph_ready.csv` (unchanged) and the seed.


In [ ]:
def make_protein_split(df, seed, train_frac=0.70, val_frac=0.15):
    rng = np.random.RandomState(seed)
    unique_proteins = df["UniProt_ID"].unique().copy()
    rng.shuffle(unique_proteins)
    n = len(unique_proteins)
    n_train, n_val = int(n * train_frac), int(n * val_frac)
    train_p = set(unique_proteins[:n_train])
    val_p   = set(unique_proteins[n_train:n_train + n_val])
    test_p  = set(unique_proteins[n_train + n_val:])
    assert train_p.isdisjoint(val_p) and train_p.isdisjoint(test_p) and val_p.isdisjoint(test_p)
    return (df[df["UniProt_ID"].isin(p)].reset_index(drop=True) for p in (train_p, val_p, test_p))


train_df, val_df, test_df = make_protein_split(df, seed=42)
print("Protein-level split is leak-free (verified disjoint by assertion above).")
for name, sub in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    frac = (sub["Target"] == 1).mean()
    print(f"{name:5s}: {len(sub):6,} variants, {sub['UniProt_ID'].nunique():4,} proteins, "
          f"{frac:.3f} pathogenic fraction")


In [ ]:
class NodeBudgetBatchSampler(Sampler):
    def __init__(self, node_counts, max_nodes_per_batch=20000, shuffle=True, seed=0):
        self.node_counts = node_counts
        self.max_nodes_per_batch = max_nodes_per_batch
        self.shuffle = shuffle
        self.seed = seed
        self._epoch = 0

    def __iter__(self):
        indices = list(range(len(self.node_counts)))
        if self.shuffle:
            r = random.Random(self.seed + self._epoch)
            r.shuffle(indices)
            self._epoch += 1
        batch, current = [], 0
        for idx in indices:
            n = self.node_counts[idx]
            if batch and current + n > self.max_nodes_per_batch:
                yield batch
                batch, current = [], 0
            batch.append(idx)
            current += n
        if batch:
            yield batch

    def __len__(self):
        return max(1, sum(self.node_counts) // self.max_nodes_per_batch)


def get_node_counts(variant_df, graph_dir):
    cache, counts = {}, []
    for uid in variant_df["UniProt_ID"]:
        if uid not in cache:
            cache[uid] = torch.load(os.path.join(graph_dir, f"{uid}.pt"), weights_only=False).num_residues
        counts.append(cache[uid])
    return counts


train_ds = MissenseVariantGraphDataset(train_df, NEW_GRAPH_DIR)
val_ds   = MissenseVariantGraphDataset(val_df, NEW_GRAPH_DIR)
test_ds  = MissenseVariantGraphDataset(test_df, NEW_GRAPH_DIR)

train_loader = DataLoader(train_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(train_df, NEW_GRAPH_DIR), 20000, shuffle=True, seed=42))
val_loader = DataLoader(val_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(val_df, NEW_GRAPH_DIR), 20000, shuffle=False))
test_loader = DataLoader(test_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(test_df, NEW_GRAPH_DIR), 20000, shuffle=False))

train_pos = (train_df["Target"] == 1).sum()
train_neg = (train_df["Target"] == 0).sum()
pos_weight_value = train_neg / train_pos
print(f"pos_weight = {pos_weight_value:.3f}")
POS_WEIGHT = torch.tensor([pos_weight_value], dtype=torch.float, device=DEVICE)


## 6. Training and Evaluation Functions
Identical methodology to Phase 4 (`run_epoch`, `train_model`, `full_metrics`, `evaluate_gnn`), same loss (BCE + pos_weight), same AUPRC-driven early stopping (patience=10, up to 60 epochs), same resumability contract (checkpoint **and** history file must both exist to skip retraining, so an interrupted run can simply be re-executed from the top without losing progress or silently reusing a partially-trained checkpoint).


In [ ]:
def run_epoch(model, loader, pos_weight, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    losses, all_logits, all_labels = [], [], []
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            batch = batch.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            logits = model(batch)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, batch.y, pos_weight=pos_weight)
            if is_train:
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            all_logits.extend(logits.detach().cpu().numpy().tolist())
            all_labels.extend(batch.y.detach().cpu().numpy().tolist())
    auprc = average_precision_score(all_labels, all_logits)
    auroc = roc_auc_score(all_labels, all_logits)
    return float(np.mean(losses)), auprc, auroc


def train_model(model, model_name, train_loader, val_loader, pos_weight, n_epochs=60, patience=10,
                lr=1e-3, weight_decay=1e-5, verbose=True, resume=True):
    checkpoint_path = os.path.join(NEW_MODELS_DIR, f"{model_name}_best.pt")
    history_path = os.path.join(EXPERIMENT_DIR, f"training_history_{model_name}.csv")

    if resume and os.path.exists(checkpoint_path) and os.path.exists(history_path):
        model.to(DEVICE)
        model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
        if verbose:
            print(f"[{model_name}] Found existing checkpoint - skipping training and reloading it. "
                  f"Delete {checkpoint_path} first if you want to retrain from scratch.")
        history_df = pd.read_csv(history_path)
        return model, history_df, history_df["val_auprc"].max()

    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_auprc = -1.0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        train_loss, train_auprc, train_auroc = run_epoch(model, train_loader, pos_weight, optimizer)
        val_loss, val_auprc, val_auroc = run_epoch(model, val_loader, pos_weight, optimizer=None)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_auprc": train_auprc,
                         "train_auroc": train_auroc, "val_loss": val_loss, "val_auprc": val_auprc,
                         "val_auroc": val_auroc})
        if verbose:
            print(f"[{model_name}] epoch {epoch:2d}: train_loss={train_loss:.4f} train_AUPRC={train_auprc:.3f} "
                  f"| val_loss={val_loss:.4f} val_AUPRC={val_auprc:.3f} val_AUROC={val_auroc:.3f}")
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    print(f"[{model_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    del optimizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return model, history_df, best_val_auprc


def plot_training_curves(history_df, model_name, safe_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title(f"{model_name}: Loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["train_auprc"], label="train AUPRC")
    axes[1].plot(history_df["epoch"], history_df["val_auprc"], label="val AUPRC")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUPRC"); axes[1].set_title(f"{model_name}: AUPRC")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(os.path.join(NEW_FIGURES_DIR, f"training_curves_{safe_name}.png"), dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def full_metrics(labels, probs, threshold=0.5):
    preds = (np.asarray(probs) >= threshold).astype(int)
    return {
        "AUPRC": average_precision_score(labels, probs),
        "AUROC": roc_auc_score(labels, probs),
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
    }


def evaluate_gnn(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            all_logits.extend(logits.cpu().numpy().tolist())
            all_labels.extend(batch.y.cpu().numpy().tolist())
    probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
    return np.array(all_labels), probs


## 7. Train GATv2 (Min-Max PAE), Default Hyperparameters
Same defaults as the project's original "default hyperparameters" pass (Phase 4 Section 7): `hidden_dim=128, heads=4, dropout=0.2, lr=1e-3, weight_decay=1e-5`. This is the fast half of the run (~7h based on the original default run's 13 logged epochs at ~31 min/epoch) and lets the notebook produce a first directional answer even if the tuned run below does not complete before you need to stop.


In [ ]:
model_minmax_default = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=True)
model_minmax_default, history_minmax_default, _ = train_model(
    model_minmax_default, "gatv2_pae_minmax_default", train_loader, val_loader, POS_WEIGHT)
plot_training_curves(history_minmax_default, "GATv2 (min-max PAE, default)", "gatv2_pae_minmax_default")


## 8. Train GATv2 (Min-Max PAE), Tuned `B_reg_small` Config
Reuses the project's existing winning hyperparameters from `best_config.json` (read-only, this notebook never re-runs or modifies the 15-hour search) so the new-transform model is tuned exactly as fairly as every other model in the project's comparison.


In [ ]:
with open(BEST_CONFIG_PATH) as f:
    best_config = json.load(f)
print(f"Loaded best_config: {best_config['name']}  hidden_dim={int(best_config['hidden_dim'])}  "
      f"dropout={best_config['dropout']:.2f}  lr={best_config['lr']}  wd={best_config['weight_decay']}")

model_minmax_tuned = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=True,
                                      hidden_dim=int(best_config["hidden_dim"]), dropout=best_config["dropout"])
model_minmax_tuned, history_minmax_tuned, _ = train_model(
    model_minmax_tuned, "gatv2_pae_minmax_tuned", train_loader, val_loader, POS_WEIGHT,
    lr=best_config["lr"], weight_decay=best_config["weight_decay"])
plot_training_curves(history_minmax_tuned, "GATv2 (min-max PAE, tuned)", "gatv2_pae_minmax_tuned")


## 9. Reload the Existing Unweighted Ablation (No Retraining)
The unweighted ablation's `forward()` never reads `edge_attr` at all when `use_edge_attr=False` (see `models.py`), its predictions cannot possibly change with a different PAE transform, because it never sees PAE in the first place. Retraining it again here would spend another ~14 hours reproducing numbers that are already known. Instead, its existing checkpoints (`models/gatv2_unweighted_best.pt`, `models/gatv2_unweighted_final_best.pt`, both read-only) are reloaded and run through inference **only**, a few minutes, not hours on this notebook's own test loader, to get properly row-aligned predictions for the paired significance tests below.


In [ ]:
model_unweighted_default = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=False)
model_unweighted_default.load_state_dict(
    torch.load(os.path.join(OLD_MODELS_DIR, "gatv2_unweighted_best.pt"), weights_only=True))
model_unweighted_default.to(DEVICE)

model_unweighted_tuned = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=False,
                                          hidden_dim=int(best_config["hidden_dim"]), dropout=best_config["dropout"])
model_unweighted_tuned.load_state_dict(
    torch.load(os.path.join(OLD_MODELS_DIR, "gatv2_unweighted_final_best.pt"), weights_only=True))
model_unweighted_tuned.to(DEVICE)

print("Reloaded both existing unweighted checkpoints (no training performed).")


## 10. Evaluate All Models on the Test Set
`test_loader` is built with `shuffle=False`, so every model below is evaluated in the exact same fixed order as `test_df`, positional alignment is verified with `assert` rather than assumed, matching the project's existing convention (Phase 4 Section 8).


In [ ]:
test_labels, probs_minmax_default = evaluate_gnn(model_minmax_default, test_loader)
test_labels_2, probs_minmax_tuned = evaluate_gnn(model_minmax_tuned, test_loader)
test_labels_3, probs_unweighted_default = evaluate_gnn(model_unweighted_default, test_loader)
test_labels_4, probs_unweighted_tuned = evaluate_gnn(model_unweighted_tuned, test_loader)
assert (np.array_equal(test_labels, test_labels_2) and np.array_equal(test_labels, test_labels_3)
        and np.array_equal(test_labels, test_labels_4)), "Label misalignment across models - do not trust results"

test_df = test_df.reset_index(drop=True)
test_df["true_label"] = test_labels.astype(int)
test_df["minmax_default_prob"] = probs_minmax_default
test_df["minmax_tuned_prob"] = probs_minmax_tuned
test_df["unweighted_default_prob"] = probs_unweighted_default
test_df["unweighted_tuned_prob"] = probs_unweighted_tuned

# Reuse cached GraphSAGE + old-transform-tuned predictions (row-aligned by VariationID) - no retraining
pred_cache = pd.read_csv(PHASE5_PRED_CACHE)
test_df = test_df.merge(
    pred_cache[["VariationID", "gatv2_prob", "sage_prob"]].rename(
        columns={"gatv2_prob": "old_transform_tuned_prob", "sage_prob": "graphsage_prob"}),
    on="VariationID", how="left")

test_df.to_csv(os.path.join(EXPERIMENT_DIR, "predictions_cache.csv"), index=False)
print(f"Cached row-level predictions for {len(test_df):,} test variants.")
print(f"GraphSAGE / old-transform match rate: "
      f"{test_df['graphsage_prob'].notna().mean():.1%} (should be ~100% - same test set)")


## 11. Combined Results Table
New models evaluated fresh in this notebook; existing baselines (GraphSAGE, XGBoost, GCN, old-transform PAE-weighted) read from the project's existing result files, untouched, not recomputed.


In [ ]:
new_rows = [
    {"model": "GATv2 (min-max PAE, default)", **full_metrics(test_labels, probs_minmax_default)},
    {"model": "GATv2 (min-max PAE, tuned)", **full_metrics(test_labels, probs_minmax_tuned)},
    {"model": "GATv2 (unweighted, default) [reloaded]", **full_metrics(test_labels, probs_unweighted_default)},
    {"model": "GATv2 (unweighted, tuned) [reloaded]", **full_metrics(test_labels, probs_unweighted_tuned)},
]

# Existing baselines, reused as-is from phase4_results_final.csv (read-only)
existing = pd.read_csv("phase4_results_final.csv")
for name in ["GraphSAGE (baseline)", "XGBoost (structure-free baseline)", "GCN (baseline)",
             "GATv2 (PAE-weighted, tuned)", "GATv2 (PAE-weighted, original)"]:
    row = existing[existing["model"] == name]
    if len(row):
        r = row.iloc[0].to_dict()
        r["model"] = r["model"] + " [existing, unchanged]"
        new_rows.append(r)

results_comparison_df = pd.DataFrame(new_rows).sort_values("AUPRC", ascending=False)
results_comparison_df.to_csv(os.path.join(EXPERIMENT_DIR, "results_comparison.csv"), index=False)
print(results_comparison_df.to_string(index=False))


## 12. Statistical Significance Testing
Identical methodology to Phase 4 Sections 9/13 (McNemar, paired bootstrap on AUPRC, DeLong on AUROC), reproduced here rather than imported so this notebook has no dependency on the other notebooks' kernel state.


In [ ]:
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2


def delong_test(probs_a, probs_b, labels):
    labels = np.asarray(labels)
    order = np.argsort(-labels)
    m = int(np.sum(labels == 1)); n = int(np.sum(labels == 0))
    preds_list = [np.asarray(probs_a), np.asarray(probs_b)]
    k = len(preds_list)
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        preds_ordered = preds_list[r][order]
        tx[r, :] = compute_midrank(preds_ordered[:m])
        ty[r, :] = compute_midrank(preds_ordered[m:])
        tz[r, :] = compute_midrank(preds_ordered)
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    delongcov = np.cov(v01) / m + np.cov(v10) / n
    assert abs(aucs[0] - roc_auc_score(labels, probs_a)) < 1e-6
    assert abs(aucs[1] - roc_auc_score(labels, probs_b)) < 1e-6
    l = np.array([1, -1])
    z = l.dot(aucs) / np.sqrt(l.dot(delongcov).dot(l))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"auc_a": aucs[0], "auc_b": aucs[1], "z": z, "p_value": p}


def paired_bootstrap_auprc(probs_a, probs_b, labels, n_boot=2000, seed=0):
    rng_b = np.random.RandomState(seed)
    labels = np.asarray(labels); probs_a = np.asarray(probs_a); probs_b = np.asarray(probs_b)
    n_test = len(labels)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng_b.randint(0, n_test, n_test)
        diffs[i] = (average_precision_score(labels[idx], probs_a[idx])
                    - average_precision_score(labels[idx], probs_b[idx]))
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs > 0).mean(), (diffs < 0).mean())
    return {"mean_diff": diffs.mean(), "ci_low": ci_low, "ci_high": ci_high, "p_value": p_value}


def mcnemar_test(probs_a, probs_b, labels, threshold=0.5):
    preds_a = (np.asarray(probs_a) >= threshold).astype(int)
    preds_b = (np.asarray(probs_b) >= threshold).astype(int)
    correct_a = preds_a == labels; correct_b = preds_b == labels
    table = [[int(np.sum(correct_a & correct_b)), int(np.sum(correct_a & ~correct_b))],
             [int(np.sum(~correct_a & correct_b)), int(np.sum(~correct_a & ~correct_b))]]
    result = mcnemar(table, exact=False, correction=True)
    return {"statistic": result.statistic, "p_value": result.pvalue}


def run_all_significance_tests(name_a, probs_a, name_b, probs_b, labels):
    mc = mcnemar_test(probs_a, probs_b, labels)
    bs = paired_bootstrap_auprc(probs_a, probs_b, labels)
    dl = delong_test(probs_a, probs_b, labels)
    significant = bs["p_value"] < 0.05 or dl["p_value"] < 0.05
    print(f"\n=== {name_a}  vs.  {name_b} ===")
    print(f"McNemar p={mc['p_value']:.4f} | Bootstrap AUPRC diff={bs['mean_diff']:+.4f} "
          f"(p={bs['p_value']:.4f}) | DeLong AUROC p={dl['p_value']:.4f} "
          f"-> {'SIGNIFICANT' if significant else 'not significant'}")
    return {"comparison": f"{name_a} vs {name_b}", "mcnemar_p": mc["p_value"],
            "bootstrap_auprc_diff": bs["mean_diff"], "bootstrap_ci_low": bs["ci_low"],
            "bootstrap_ci_high": bs["ci_high"], "bootstrap_p": bs["p_value"],
            "delong_z": dl["z"], "delong_p": dl["p_value"], "significant": significant}


sig_rows = []
sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, default)", probs_minmax_default,
    "GATv2 (unweighted, default)", probs_unweighted_default, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, tuned)", probs_minmax_tuned,
    "GATv2 (unweighted, tuned)", probs_unweighted_tuned, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, tuned)", probs_minmax_tuned,
    "GATv2 (old transform 1/(1+PAE), tuned)", test_df["old_transform_tuned_prob"].values, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, tuned)", probs_minmax_tuned,
    "GraphSAGE (baseline)", test_df["graphsage_prob"].values, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, default)", probs_minmax_default,
    "GATv2 (min-max PAE, tuned)", probs_minmax_tuned, test_labels))

pd.DataFrame(sig_rows).to_csv(os.path.join(EXPERIMENT_DIR, "significance_tests.csv"), index=False)
print("\nSaved significance_tests.csv")


## 13. PAE Value Quantification (Proposal-Required Deliverable, This Transform)
The same %-change-in-AUPRC deliverable computed for the original transform in `pae_value_quantification.csv`, recomputed here for the min-max transform so the two are directly comparable.


In [ ]:
def pct_row(stage, pae_probs, unweighted_probs, labels, sig_row):
    pae_auprc = average_precision_score(labels, pae_probs)
    uw_auprc = average_precision_score(labels, unweighted_probs)
    pct = (pae_auprc - uw_auprc) / uw_auprc * 100
    return {"Stage": stage, "PAE-weighted AUPRC": round(pae_auprc, 4),
            "Unweighted AUPRC": round(uw_auprc, 4), "% change (PAE vs unweighted)": round(pct, 3),
            "bootstrap_p": round(sig_row["bootstrap_p"], 4), "delong_p": round(sig_row["delong_p"], 4),
            "significant (alpha=0.05)": bool(sig_row["significant"])}


pae_quant_rows = [
    pct_row("Default (untuned)", probs_minmax_default, probs_unweighted_default, test_labels, sig_rows[0]),
    pct_row("Tuned (B_reg_small)", probs_minmax_tuned, probs_unweighted_tuned, test_labels, sig_rows[1]),
]
pae_quant_df = pd.DataFrame(pae_quant_rows)
pae_quant_df.to_csv(os.path.join(EXPERIMENT_DIR, "pae_value_quantification_minmax.csv"), index=False)
print(pae_quant_df.to_string(index=False))


## 14. AlphaMissense Head-to-Head Comparison
Same methodology as Phase 5, reusing the existing cached AlphaMissense scores (`alphamissense_test_scores.csv`, read-only, no re-download) and the cached GraphSAGE predictions, joined against this notebook's own min-max-tuned predictions.


In [ ]:
AA3TO1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C",
    "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H", "Ile": "I",
    "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P",
    "Ser": "S", "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
}
test_df["ref_1"] = test_df["Ref_AA"].map(AA3TO1)
test_df["variant_key"] = test_df["ref_1"] + test_df["Pos"].astype(str) + test_df["Alt_AA_1"]

am_scores = pd.read_csv(AM_CACHE_PATH)
am_clean = am_scores.rename(columns={"uniprot_id": "UniProt_ID", "protein_variant": "variant_key"})[
    ["UniProt_ID", "variant_key", "am_pathogenicity", "am_class"]]

merged = test_df.merge(am_clean, on=["UniProt_ID", "variant_key"], how="inner")
coverage = len(merged) / len(test_df) * 100
print(f"Coverage: {len(merged):,} / {len(test_df):,} test variants matched ({coverage:.1f}%)")

AM_THRESHOLD = 0.564
am_labels = merged["true_label"].values.astype(int)

def metrics_row(name, probs, labels, threshold=0.5):
    m = full_metrics(labels, probs, threshold)
    m["Model"] = name
    m["Threshold"] = threshold
    return m

am_results = [
    metrics_row("GATv2 (min-max PAE, tuned)", merged["minmax_tuned_prob"].values, am_labels, 0.5),
    metrics_row("GraphSAGE (baseline)", merged["graphsage_prob"].values, am_labels, 0.5),
    metrics_row("AlphaMissense", merged["am_pathogenicity"].values, am_labels, AM_THRESHOLD),
]
am_results_df = pd.DataFrame(am_results)[["Model", "AUPRC", "AUROC", "Accuracy", "Precision", "Recall", "F1", "Threshold"]]
am_results_df = am_results_df.sort_values("AUPRC", ascending=False)
am_results_df.to_csv(os.path.join(EXPERIMENT_DIR, "alphamissense_comparison_results.csv"), index=False)
print(am_results_df.to_string(index=False))

am_sig_rows = []
am_sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, tuned)", merged["minmax_tuned_prob"].values,
    "AlphaMissense", merged["am_pathogenicity"].values, am_labels))
am_sig_rows.append(run_all_significance_tests(
    "GATv2 (min-max PAE, tuned)", merged["minmax_tuned_prob"].values,
    "GraphSAGE (baseline)", merged["graphsage_prob"].values, am_labels))
am_sig_rows.append(run_all_significance_tests(
    "GraphSAGE (baseline)", merged["graphsage_prob"].values,
    "AlphaMissense", merged["am_pathogenicity"].values, am_labels))
pd.DataFrame(am_sig_rows).to_csv(os.path.join(EXPERIMENT_DIR, "alphamissense_significance_tests.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = ["GATv2\n(min-max PAE,\ntuned)", "GraphSAGE\n(baseline)", "AlphaMissense"]
colors = ["#2E6F8E", "#4CAF50", "#FF5722"]
for ax, metric in zip(axes, ["AUPRC", "AUROC"]):
    vals = [am_results_df.set_index("Model").loc[n, metric] for n in
            ["GATv2 (min-max PAE, tuned)", "GraphSAGE (baseline)", "AlphaMissense"]]
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="white")
    ax.set_ylim(min(vals) - 0.02, max(vals) + 0.03)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} - matched subset (n={len(merged):,})")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001, f"{val:.4f}",
                ha="center", va="bottom", fontsize=9)
plt.suptitle("Min-max PAE transform vs. GraphSAGE vs. AlphaMissense", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(NEW_FIGURES_DIR, "alphamissense_comparison_minmax.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


## 15. Final Summary


In [ ]:
print("="*70)
print("SUMMARY: Per-Protein Min-Max PAE Transform vs. Original 1/(1+PAE)")
print("="*70)
print(f"\nSaturation check (300-protein sample, see Section 4):")
print(f"  OLD p50/p75/p95/p99 all ~0.5 (saturated)")
print(f"  NEW p50={np.percentile(new_weights_all,50):.3f} p75={np.percentile(new_weights_all,75):.3f} "
      f"p95={np.percentile(new_weights_all,95):.3f} p99={np.percentile(new_weights_all,99):.3f}")
print(f"\nModel comparison (full table in results_comparison.csv):")
print(results_comparison_df[["model", "AUPRC", "AUROC", "Accuracy"]].to_string(index=False))
print(f"\nPAE Value Quantification (full table in pae_value_quantification_minmax.csv):")
print(pae_quant_df.to_string(index=False))
print(f"\nAlphaMissense comparison (full table in alphamissense_comparison_results.csv):")
print(am_results_df.to_string(index=False))


### Methods note (copy-paste for the paper's robustness-check / limitations section)
> As a robustness check on the `1/(1+PAE)` edge-weight transform's saturation (diagnosed in Section 2 of the main training notebook), an alternative per-protein min-max normalization was tested: `weight = 1 - (PAE - min) / (max - min)`, computed independently over each protein's own contact-edge PAE values, so every protein uses the full [0, 1] weight range regardless of its absolute PAE scale. The GATv2 architecture, training procedure, hyperparameter configurations, and train/val/test split were kept identical to the main experiment; only the PAE-to-weight transform changed. [Fill in after running: whether this changed the PAE-weighted-vs-unweighted comparison's direction or significance.]
